
##problems
###Duplicates	12 duplicate transactions
###Inconsistent categories	Deposit, withdrawal, DEPOSIT, etc.
###Missing amounts	15 missing values
###Invalid amounts	5 values recorded as N/A
###Suspicious amounts	5 transactions of KES 250,000
###Date inconsistency	5 dates use / instead of -
###Unknown agents	8 records reference AG099

Uniqueness and consistency
- Identified duplicate transactions using the txn_id column.
- Removed duplicate transaction IDs while retaining the first occurrence.
- Standardised transaction types by removing unnecessary spaces and converting text to lowercase before mapping to the required categories.
- The final transaction types are Deposit and Withdrawal.

In [ ]:

# ============================================================
# AGENT TRANSACTIONS DATA CLEANING AND ANALYSIS
# ============================================================

# 1. IMPORT LIBRARIES

import numpy as np
import pandas as pd


# ============================================================
# 2. GENERATE AGENT DATASET
# ============================================================

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"],
        20
    ),
    "onboarded": pd.date_range(
        "2024-01-01",
        periods=20,
        freq="MS"
    ).strftime("%Y-%m-%d")
})


# ============================================================
# 3. GENERATE TRANSACTION DATASET
# ============================================================

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],

    "agent_id": rng.choice(agents["agent_id"], n),

    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ",
         "Withdrawal", "deposit"],
        n
    ),

    "amount": rng.lognormal(
        7.5, 0.8, n
    ).round(0).astype(str),

    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(
            rng.integers(0, 60, n),
            unit="D"
        )
    ).strftime("%Y-%m-%d")
})


# ============================================================
# 4. INTRODUCE DATA QUALITY PROBLEMS
# ============================================================

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan

txns.loc[idx[15:20], "amount"] = "N/A"

txns.loc[idx[20:25], "amount"] = "250000"

txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"]
    .str.replace("-", "/")
)

# Introduce duplicate transactions

txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

# Introduce unknown agents

txns.loc[
    rng.choice(len(txns), 8, replace=False),
    "agent_id"
] = "AG099"


# ============================================================
# 5. SAVE AND LOAD CSV FILES
# ============================================================

agents.to_csv("agents.csv", index=False)

txns.to_csv("agent_transactions.csv", index=False)

agents = pd.read_csv("agents.csv")

txns = pd.read_csv("agent_transactions.csv")


# ============================================================
# 6. DISPLAY DATASETS
# ============================================================

print("AGENTS DATASET")
display(agents.head())

print("TRANSACTIONS DATASET")
display(txns.head())


# ============================================================
# 7. EXPLORE DATASETS
# ============================================================

print("Agents shape:", agents.shape)

print("Transactions shape:", txns.shape)

print("\nAgent data types:")
print(agents.dtypes)

print("\nTransaction data types:")
print(txns.dtypes)

print("\nMissing values in agents:")
print(agents.isna().sum())

print("\nMissing values in transactions:")
print(txns.isna().sum())


# ============================================================
# 8. IDENTIFY AND REMOVE DUPLICATES
# ============================================================

print(
    "Duplicate transaction IDs:",
    txns["txn_id"].duplicated().sum()
)

txns = txns.drop_duplicates(
    subset="txn_id",
    keep="first"
).copy()

print("Rows after removing duplicates:", len(txns))

print(
    "Remaining duplicate IDs:",
    txns["txn_id"].duplicated().sum()
)


# ============================================================
# 9. CLEAN TRANSACTION TYPES
# ============================================================

print("\nOriginal transaction types:")

print(txns["txn_type"].value_counts(dropna=False))

txns["txn_type"] = (
    txns["txn_type"]
    .str.strip()
    .str.lower()
    .map({
        "deposit": "Deposit",
        "withdrawal": "Withdrawal"
    })
)

print("\nCleaned transaction types:")

print(txns["txn_type"].value_counts(dropna=False))

print("\nMissing transaction types:")

print(txns["txn_type"].isna().sum())

print("\nUnique transaction types:")

print(txns["txn_type"].unique())


# ============================================================
# 10. CLEAN TRANSACTION AMOUNTS AND DATES
# ============================================================

clean_txns = txns.copy()

clean_txns["amount"] = pd.to_numeric(
    clean_txns["amount"],
    errors="coerce"
)

clean_txns["txn_date"] = pd.to_datetime(
    clean_txns["txn_date"],
    errors="coerce",
    format="mixed"
)

print("\nCleaned data types:")

print(clean_txns.dtypes)

print("\nInvalid dates:")

print(clean_txns["txn_date"].isna().sum())


# ============================================================
# 11. IDENTIFY MISSING AMOUNTS
# ============================================================

clean_txns["amount_missing"] = (
    clean_txns["amount"].isna()
)

print("\nMissing amount indicators:")

print(clean_txns["amount_missing"].value_counts())

print(
    "Total missing amounts:",
    clean_txns["amount"].isna().sum()
)


# ============================================================
# 12. DETECT OUTLIERS USING IQR
# ============================================================

Q1 = clean_txns["amount"].quantile(0.25)

Q3 = clean_txns["amount"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - (1.5 * IQR)

upper_limit = Q3 + (1.5 * IQR)

clean_txns["is_outlier"] = (
    (clean_txns["amount"] < lower_limit) |
    (clean_txns["amount"] > upper_limit)
)

print("\nQ1:", Q1)

print("Q3:", Q3)

print("IQR:", IQR)

print("Lower limit:", lower_limit)

print("Upper limit:", upper_limit)

print(
    "Total outliers:",
    clean_txns["is_outlier"].sum()
)


# ============================================================
# 13. INVESTIGATE HIGH TRANSACTIONS
# ============================================================

high_transactions = clean_txns[
    clean_txns["amount"] == 250000
]

print("\nTransactions with amount 250000:")

display(high_transactions)


# ============================================================
# 14. IDENTIFY UNKNOWN AGENTS
# ============================================================

clean_txns = clean_txns.merge(
    agents[["agent_id", "county"]],
    on="agent_id",
    how="left",
    indicator=True
)

print("\nAgent matching results:")

print(clean_txns["_merge"].value_counts())

unknown_agents = clean_txns[
    clean_txns["_merge"] == "left_only"
]

print(
    "\nUnknown agent transactions:",
    len(unknown_agents)
)

display(unknown_agents)

clean_txns["unknown_agent"] = (
    clean_txns["_merge"] == "left_only"
)

clean_txns["county"] = (
    clean_txns["county"].fillna("Unknown")
)

clean_txns = clean_txns.drop(
    columns="_merge"
)


# ============================================================
# 15. COUNTY TRANSACTION SUMMARY
# ============================================================

county_summary = (
    clean_txns
    .groupby(["county", "txn_type"])
    .agg(
        total_value=("amount", "sum"),
        number_of_transactions=("txn_id", "count"),
        average_value=("amount", "mean")
    )
    .reset_index()
)

print("\nCOUNTY TRANSACTION SUMMARY:")

display(county_summary)


# ============================================================
# 16. WEEKLY TRANSACTION SUMMARY
# ============================================================

clean_txns["week"] = (
    clean_txns["txn_date"]
    .dt.to_period("W")
    .dt.start_time
)

weekly_summary = pd.pivot_table(
    clean_txns,
    values="amount",
    index="week",
    columns="county",
    aggfunc="sum",
    fill_value=0
)

print("\nWEEKLY TRANSACTION SUMMARY:")

display(weekly_summary)


# ============================================================
# 17. DATA QUALITY REPORT
# ============================================================

data_quality_report = pd.DataFrame({
    "Issue": [
        "Duplicate transactions",
        "Missing transaction amounts",
        "Invalid transaction types",
        "Invalid transaction dates",
        "Outlier transactions",
        "Unknown agent transactions"
    ],

    "Count": [
        12,
        clean_txns["amount_missing"].sum(),
        clean_txns["txn_type"].isna().sum(),
        clean_txns["txn_date"].isna().sum(),
        clean_txns["is_outlier"].sum(),
        clean_txns["unknown_agent"].sum()
    ]
})

print("\nDATA QUALITY REPORT:")

display(data_quality_report)


# ============================================================
# 18. EXPORT CLEANED DATA AND SUMMARIES
# ============================================================

clean_txns.to_csv(
    "cleaned_agent_transactions.csv",
    index=False
)

county_summary.to_csv(
    "county_summary.csv",
    index=False
)

weekly_summary.to_csv(
    "weekly_summary.csv"
)

data_quality_report.to_csv(
    "data_quality_report.csv",
    index=False
)


# ============================================================
# 19. FINAL VERIFICATION
# ============================================================

print("\nFINAL DATASET SHAPE:")

print(clean_txns.shape)

print("\nFINAL DATA TYPES:")

print(clean_txns.dtypes)

print("\nMISSING VALUES:")

print(clean_txns.isna().sum())

print("\nDUPLICATE TRANSACTION IDs:")

print(clean_txns["txn_id"].duplicated().sum())

print("\nTRANSACTION TYPES:")

print(clean_txns["txn_type"].value_counts(dropna=False))

print("\nCOUNTIES:")

print(clean_txns["county"].value_counts())

print("\nFIRST FIVE CLEANED TRANSACTIONS:")

display(clean_txns.head())


# ============================================================
# 20. CONFIRM EXPORTED FILES
# ============================================================

import os

print("\nEXPORTED FILES:")

print(os.listdir())


In [ ]:
clean_txns = txns.copy()
clean_txns["amount"] = pd.to_numeric(
    clean_txns["amount"],
    errors="coerce"
)
clean_txns["txn_date"] = pd.to_datetime(
    clean_txns["txn_date"],
    errors="coerce",
    format="mixed"
)
print(clean_txns.dtypes)

print("\nInvalid dates:")
print(clean_txns["txn_date"].isna().sum())
clean_txns["amount_missing"] = clean_txns["amount"].isna()

print(clean_txns["amount_missing"].value_counts())
print("Missing amounts:",
      clean_txns["amount"].isna().sum())
Q1 = clean_txns["amount"].quantile(0.25)
Q3 = clean_txns["amount"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - (1.5 * IQR)
upper_limit = Q3 + (1.5 * IQR)

clean_txns["is_outlier"] = (
    (clean_txns["amount"] < lower_limit) |
    (clean_txns["amount"] > upper_limit)
)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

print("Total outliers:",
      clean_txns["is_outlier"].sum())
high_transactions = clean_txns[
    clean_txns["amount"] == 250000
]

display(high_transactions)
clean_txns = clean_txns.merge(
    agents[["agent_id", "county"]],
    on="agent_id",
    how="left",
    indicator=True
)
print(clean_txns["_merge"].value_counts())
unknown_agents = clean_txns[
    clean_txns["_merge"] == "left_only"
]

print("Unknown agent transactions:",
      len(unknown_agents))

display(unknown_agents)
clean_txns["unknown_agent"] = (
    clean_txns["_merge"] == "left_only"
)

clean_txns["county"] = (
    clean_txns["county"].fillna("Unknown")
)

clean_txns = clean_txns.drop(columns="_merge")
county_summary = (
    clean_txns
    .groupby(["county", "txn_type"])
    .agg(
        total_value=("amount", "sum"),
        number_of_transactions=("txn_id", "count"),
        average_value=("amount", "mean")
    )
    .reset_index()
)

display(county_summary)
clean_txns["week"] = (
    clean_txns["txn_date"]
    .dt.to_period("W")
    .dt.start_time
)
weekly_summary = pd.pivot_table(
    clean_txns,
    values="amount",
    index="week",
    columns="county",
    aggfunc="sum",
    fill_value=0
)



txn_id              object
agent_id            object
txn_type            object
amount             float64
txn_date    datetime64[ns]
dtype: object

Invalid dates:
0
amount_missing
False    580
True      20
Name: count, dtype: int64
Missing amounts: 20
Q1: 1077.0
Q3: 3080.0
IQR: 2003.0
Lower limit: -1927.5
Upper limit: 6084.5
Total outliers: 40


,txn_id,agent_id,txn_type,amount,txn_date,amount_missing,is_outlier
306,TX00307,AG012,Deposit,250000.0,2026-03-02,False,True
315,TX00316,AG014,Deposit,250000.0,2026-04-01,False,True
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02,False,True
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11,False,True
594,TX00595,AG016,Deposit,250000.0,2026-03-11,False,True


_merge
both          593
left_only       7
right_only      0
Name: count, dtype: int64
Unknown agent transactions: 7


,txn_id,agent_id,txn_type,amount,txn_date,amount_missing,is_outlier,county,_merge
55,TX00056,AG099,Deposit,8507.0,2026-04-01,False,True,NaN,left_only
68,TX00069,AG099,Withdrawal,1877.0,2026-04-24,False,False,NaN,left_only
77,TX00078,AG099,Deposit,1770.0,2026-04-06,False,False,NaN,left_only
237,TX00238,AG099,Deposit,4944.0,2026-04-06,False,False,NaN,left_only
267,TX00268,AG099,Deposit,3110.0,2026-04-09,False,False,NaN,left_only
444,TX00445,AG099,Deposit,5309.0,2026-04-27,False,False,NaN,left_only
557,TX00558,AG099,Deposit,1639.0,2026-03-22,False,False,NaN,left_only


,county,txn_type,total_value,number_of_transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,75274.0,27,2895.153846
2,Kisumu,Deposit,431091.0,74,6071.704225
3,Kisumu,Withdrawal,74598.0,37,2072.166667
4,Mombasa,Deposit,141880.0,72,2086.470588
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,536484.0,118,4706.000000
7,Nairobi,Withdrawal,154930.0,62,2671.206897
8,Nakuru,Deposit,181337.0,60,3126.500000
9,Nakuru,Withdrawal,326818.0,39,8600.473684


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,967.0,0.0
2026-03-02,18317.0,287933.0,283893.0,68967.0,56574.0,0.0
2026-03-09,265995.0,23186.0,28701.0,51084.0,28730.0,0.0
2026-03-16,22054.0,24572.0,23368.0,41331.0,39795.0,1639.0
2026-03-23,11504.0,27077.0,17050.0,39597.0,17222.0,0.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06,15030.0,15341.0,45316.0,55010.0,276609.0,9824.0
2026-04-13,12265.0,25863.0,26730.0,54656.0,24376.0,0.0
2026-04-20,24507.0,37439.0,16687.0,61634.0,30862.0,1877.0


## Conclusion

The project demonstrated the process of cleaning and analysing agent transaction data using Python, Pandas and NumPy.

The initial exploration helped identify data quality issues, including duplicate records, missing amounts, inconsistent transaction types, invalid date formats, unusually high transaction values and transactions associated with unknown agents.

These issues were investigated and addressed through data-cleaning techniques such as duplicate removal, text standardisation, data type conversion, missing-value identification, outlier detection and dataset merging.

The analysis also produced county-level and weekly transaction summaries, providing a clearer understanding of transaction activity across different locations and time periods.

Finally, the cleaned dataset and analysis summaries were exported into CSV files for future use.

Overall, this project demonstrates how data cleaning and exploratory data analysis can improve data consistency, reliability and usefulness for decision-making.

In [ ]:

# =====================================================
# AGENT TRANSACTIONS DATA CLEANING AND ANALYSIS
# =====================================================

# 1. IMPORT LIBRARIES

import numpy as np
import pandas as pd


# 2. GENERATE AGENT DATASET

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"],
        20
    ),
    "onboarded": pd.date_range(
        "2024-01-01",
        periods=20,
        freq="MS"
    ).strftime("%Y-%m-%d")
})


# 3. GENERATE TRANSACTION DATASET

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],

    "agent_id": rng.choice(agents["agent_id"], n),

    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ",
         "Withdrawal", "deposit"],
        n
    ),

    "amount": rng.lognormal(
        7.5, 0.8, n
    ).round(0).astype(str),

    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(
            rng.integers(0, 60, n),
            unit="D"
        )
    ).strftime("%Y-%m-%d")
})


# 4. INTRODUCE DATA QUALITY ISSUES

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan

txns.loc[idx[15:20], "amount"] = "N/A"

txns.loc[idx[20:25], "amount"] = "250000"

txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"]
    .str.replace("-", "/")
)

# Introduce duplicate transactions

txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

# Introduce unknown agents

txns.loc[
    rng.choice(len(txns), 8, replace=False),
    "agent_id"
] = "AG099"


# 5. SAVE AND LOAD DATASETS

agents.to_csv("agents.csv", index=False)

txns.to_csv("agent_transactions.csv", index=False)

agents = pd.read_csv("agents.csv")

txns = pd.read_csv("agent_transactions.csv")


# 6. DISPLAY DATASETS

print("AGENTS DATASET")
display(agents.head())

print("TRANSACTIONS DATASET")
display(txns.head())


# 7. EXPLORE DATASETS

print("Agents shape:", agents.shape)

print("Transactions shape:", txns.shape)

print("\nAgent data types:")
print(agents.dtypes)

print("\nTransaction data types:")
print(txns.dtypes)

print("\nMissing values in agents:")
print(agents.isna().sum())

print("\nMissing values in transactions:")
print(txns.isna().sum())


# 8. REMOVE DUPLICATES

print(
    "Duplicate transaction IDs:",
    txns["txn_id"].duplicated().sum()
)

txns = txns.drop_duplicates(
    subset="txn_id",
    keep="first"
).copy()

print("Rows after removing duplicates:", len(txns))

print(
    "Remaining duplicate IDs:",
    txns["txn_id"].duplicated().sum()
)


# 9. CLEAN TRANSACTION TYPES

print("\nOriginal transaction types:")

print(txns["txn_type"].value_counts(dropna=False))

txns["txn_type"] = (
    txns["txn_type"]
    .str.strip()
    .str.lower()
    .map({
        "deposit": "Deposit",
        "withdrawal": "Withdrawal"
    })
)

print("\nCleaned transaction types:")

print(txns["txn_type"].value_counts(dropna=False))

print("\nMissing transaction types:")

print(txns["txn_type"].isna().sum())

print("\nUnique transaction types:")

print(txns["txn_type"].unique())


# 10. CONVERT AMOUNTS AND DATES

clean_txns = txns.copy()

clean_txns["amount"] = pd.to_numeric(
    clean_txns["amount"],
    errors="coerce"
)

clean_txns["txn_date"] = pd.to_datetime(
    clean_txns["txn_date"],
    errors="coerce",
    format="mixed"
)

print("\nCleaned data types:")

print(clean_txns.dtypes)

print("\nInvalid dates:")

print(clean_txns["txn_date"].isna().sum())


# 11. IDENTIFY MISSING AMOUNTS

clean_txns["amount_missing"] = (
    clean_txns["amount"].isna()
)

print("\nMissing amount indicators:")

print(clean_txns["amount_missing"].value_counts())

print(
    "Missing amounts:",
    clean_txns["amount"].isna().sum()
)


# 12. DETECT OUTLIERS USING IQR

Q1 = clean_txns["amount"].quantile(0.25)

Q3 = clean_txns["amount"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - (1.5 * IQR)

upper_limit = Q3 + (1.5 * IQR)

clean_txns["is_outlier"] = (
    (clean_txns["amount"] < lower_limit) |
    (clean_txns["amount"] > upper_limit)
)

print("\nQ1:", Q1)

print("Q3:", Q3)

print("IQR:", IQR)

print("Lower limit:", lower_limit)

print("Upper limit:", upper_limit)

print(
    "Total outliers:",
    clean_txns["is_outlier"].sum()
)


# 13. INVESTIGATE HIGH TRANSACTIONS

high_transactions = clean_txns[
    clean_txns["amount"] == 250000
]

print("\nHigh transactions:")

display(high_transactions)


# 14. IDENTIFY UNKNOWN AGENTS

clean_txns = clean_txns.merge(
    agents[["agent_id", "county"]],
    on="agent_id",
    how="left",
    indicator=True
)

print("\nAgent matching results:")

print(clean_txns["_merge"].value_counts())

unknown_agents = clean_txns[
    clean_txns["_merge"] == "left_only"
]

print(
    "\nUnknown agent transactions:",
    len(unknown_agents)
)

display(unknown_agents)

clean_txns["unknown_agent"] = (
    clean_txns["_merge"] == "left_only"
)

clean_txns["county"] = (
    clean_txns["county"].fillna("Unknown")
)

clean_txns = clean_txns.drop(columns="_merge")


# 15. COUNTY TRANSACTION SUMMARY

county_summary = (
    clean_txns
    .groupby(["county", "txn_type"])
    .agg(
        total_value=("amount", "sum"),
        number_of_transactions=("txn_id", "count"),
        average_value=("amount", "mean")
    )
    .reset_index()
)

print("\nCOUNTY TRANSACTION SUMMARY:")

display(county_summary)


# 16. WEEKLY TRANSACTION SUMMARY

clean_txns["week"] = (
    clean_txns["txn_date"]
    .dt.to_period("W")
    .dt.start_time
)

weekly_summary = pd.pivot_table(
    clean_txns,
    values="amount",
    index="week",
    columns="county",
    aggfunc="sum",
    fill_value=0
)

print("\nWEEKLY TRANSACTION SUMMARY:")

display(weekly_summary)


# =====================================================
# END OF CURRENT ANALYSIS
# =====================================================


### Identifying Missing Transaction Amounts

A new Boolean column named `amount_missing` is created to identify transactions with missing amounts.

The `isna()` method checks for missing values and returns True for missing amounts and False for available amounts.

The total number of missing transaction amounts is then calculated.

In [ ]:
clean_txns["amount_missing"] = (
    clean_txns["amount"].isna()
)

print(clean_txns["amount_missing"].value_counts())

print(
    "Missing amounts:",
    clean_txns["amount"].isna().sum()
)

NameError: name 'clean_txns' is not defined